In [1]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)

# 1. Generate raw Engagement 2 CSVs (seed 44)
rng = np.random.default_rng(44)
n = 600
agents_raw = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})
txns_raw = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents_raw["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})
idx = rng.choice(n, 30, replace=False)
txns_raw.loc[idx[:15], "amount"] = np.nan
txns_raw.loc[idx[15:20], "amount"] = "N/A"
txns_raw.loc[idx[20:25], "amount"] = "250000"
txns_raw.loc[idx[25:], "txn_date"] = txns_raw.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns_raw = pd.concat([txns_raw, txns_raw.sample(12, random_state=1)], ignore_index=True)
txns_raw.loc[rng.choice(len(txns_raw), 8, replace=False), "agent_id"] = "AG099"

agents_raw.to_csv("agents.csv", index=False)
txns_raw.to_csv("agent_transactions.csv", index=False)

# 2. Write cleaning_pipeline.py
pipeline_code = '''"""Reusable data-cleaning pipeline for mobile money agent transactions."""

from __future__ import annotations

import numpy as np
import pandas as pd


def clean_agent_data(
    txns: pd.DataFrame, agents: pd.DataFrame
) -> tuple[pd.DataFrame, dict]:
    """Clean raw mobile money agent transactions and merge with agent metadata.

    Applies deterministic, rule-based cleaning without hard-coded row indices:
    1. Audits data issues across six dimensions (Uniqueness, Consistency, Validity,
       Completeness, Accuracy, Timeliness).
    2. Removes duplicate `txn_id` records (keeping first occurrence).
    3. Standardises `txn_type` to Title Case ("Deposit" or "Withdrawal").
    4. Parses `amount` to numeric, flags missing/non-numeric values via
       `amount_imputed`, and imputes using per-`txn_type` median (fallback overall median).
    5. Parses `txn_date` to ISO format (`YYYY-MM-DD`).
    6. Flags extreme amounts outside `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]` via `is_outlier`.
    7. Left-joins `agents` on `agent_id` and labels unmatched agents as `"Unknown"`.

    Returns
    -------
    tuple[pd.DataFrame, dict]
        `(clean_df, log)` where `clean_df` is the cleaned transaction DataFrame and
        `log` records audit findings, counts, imputation medians, IQR fences, and merge stats.
    """
    raw_rows = len(txns)
    raw_numeric = pd.to_numeric(txns["amount"], errors="coerce")
    raw_dates = pd.to_datetime(txns["txn_date"], format="mixed", errors="coerce")

    dup_extra_rows = int(txns.duplicated(subset=["txn_id"]).sum())
    inconsistent_type_rows = int(
        (~txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum()
    )
    slash_date_rows = int(txns["txn_date"].astype(str).str.contains("/", na=False).sum())
    unparseable_date_rows = int(raw_dates.isna().sum())
    missing_na_amount_rows = int(txns["amount"].isna().sum())
    non_numeric_str_amount_rows = int(
        (txns["amount"].notna() & raw_numeric.isna()).sum()
    )
    high_value_250k_rows = int((raw_numeric == 250000).sum())
    unknown_agent_raw_rows = int((~txns["agent_id"].isin(agents["agent_id"])).sum())

    audit_df = pd.DataFrame([
        {
            "issue": "Duplicate txn_id rows",
            "dimension": "Uniqueness",
            "column": "txn_id",
            "rows_affected": dup_extra_rows,
        },
        {
            "issue": "Non-standard casing or trailing whitespace in txn_type",
            "dimension": "Consistency",
            "column": "txn_type",
            "rows_affected": inconsistent_type_rows,
        },
        {
            "issue": "Slash date delimiter (YYYY/MM/DD instead of YYYY-MM-DD)",
            "dimension": "Validity",
            "column": "txn_date",
            "rows_affected": slash_date_rows,
        },
        {
            "issue": "Missing (NaN) transaction amounts",
            "dimension": "Completeness",
            "column": "amount",
            "rows_affected": missing_na_amount_rows,
        },
        {
            "issue": "Non-numeric text (\\x27N/A\\x27) in transaction amounts",
            "dimension": "Validity",
            "column": "amount",
            "rows_affected": non_numeric_str_amount_rows,
        },
        {
            "issue": "Extreme KES 250,000 transaction amounts",
            "dimension": "Accuracy",
            "column": "amount",
            "rows_affected": high_value_250k_rows,
        },
        {
            "issue": "Unregistered agent_id (\\x27AG099\\x27) not in agents register",
            "dimension": "Consistency",
            "column": "agent_id",
            "rows_affected": unknown_agent_raw_rows,
        },
    ])

    df = txns.drop_duplicates(subset=["txn_id"], keep="first").copy()
    rows_after_dedup = len(df)

    df["txn_type"] = df["txn_type"].astype(str).str.strip().str.capitalize()

    parsed_amt = pd.to_numeric(df["amount"], errors="coerce")
    df["amount_imputed"] = parsed_amt.isna()

    medians_by_type = (
        pd.DataFrame({"txn_type": df["txn_type"], "amt": parsed_amt})
        .groupby("txn_type")["amt"]
        .median()
        .to_dict()
    )
    overall_median = float(parsed_amt.median())
    df["amount"] = (
        parsed_amt.fillna(df["txn_type"].map(medians_by_type))
        .fillna(overall_median)
        .astype(float)
    )

    parsed_dt = pd.to_datetime(df["txn_date"], format="mixed", errors="coerce")
    df["txn_date"] = parsed_dt.dt.strftime("%Y-%m-%d")

    q1 = float(df["amount"].quantile(0.25))
    q3 = float(df["amount"].quantile(0.75))
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    df["is_outlier"] = (df["amount"] < lower_fence) | (df["amount"] > upper_fence)

    unknown_agents_clean = int((~df["agent_id"].isin(agents["agent_id"])).sum())
    agent_lookup = agents[["agent_id", "county"]].drop_duplicates(subset=["agent_id"])
    df = df.merge(agent_lookup, on="agent_id", how="left")
    df["county"] = df["county"].fillna("Unknown")

    clean_df = df[[
        "txn_id",
        "agent_id",
        "county",
        "txn_type",
        "amount",
        "txn_date",
        "amount_imputed",
        "is_outlier",
    ]].copy()

    log = {
        "raw_rows": raw_rows,
        "clean_rows": len(clean_df),
        "rows_after_dedup": rows_after_dedup,
        "duplicates_removed": dup_extra_rows,
        "imputed_rows": int(clean_df["amount_imputed"].sum()),
        "medians_by_type": medians_by_type,
        "overall_median": overall_median,
        "iqr_bounds": {
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_fence": lower_fence,
            "upper_fence": upper_fence,
        },
        "outlier_rows": int(clean_df["is_outlier"].sum()),
        "unknown_agent_txns_raw": unknown_agent_raw_rows,
        "unknown_agent_txns_clean": unknown_agents_clean,
        "unparseable_date_rows": unparseable_date_rows,
        "audit_table": audit_df,
    }
    return clean_df, log
'''

with open("cleaning_pipeline.py", "w") as f:
    f.write(pipeline_code.strip() + "\n")

print("Generated agents.csv, agent_transactions.csv, and cleaning_pipeline.py")

Generated agents.csv, agent_transactions.csv, and cleaning_pipeline.py


# Mobile Money Agent Network — Data Audit, Cleaning & Summary
## 1. Data-Quality Audit & Pipeline Execution (`clean_agent_data`)

In [2]:
import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

# Run pipeline and save cleaned CSV
clean_txns, log = clean_agent_data(txns, agents)
clean_txns.to_csv("outputs/agent_transactions_clean.csv", index=False)

# Data-quality audit table as a DataFrame (issue, dimension, column, rows_affected)
audit_df = log["audit_table"]
print("=== Data-Quality Audit Table ===")
display(audit_df)

# Post-merge validation: row counts and unknown agents
print("\n=== Post-Merge Validation ===")
print(f"Raw transactions rows:     {len(txns)}")
print(f"Cleaned transactions rows: {len(clean_txns)} (duplicates removed: {log['duplicates_removed']})")
print(f"Unknown agent txns (raw):  {log['unknown_agent_txns_raw']}")
print(f"Unknown agent txns (clean):{log['unknown_agent_txns_clean']} (assigned county='Unknown')")
print(f"Imputed amount rows:       {log['imputed_rows']} | Medians by type: {log['medians_by_type']}")
print(f"IQR upper fence:           {log['iqr_bounds']['upper_fence']} | Flagged outliers: {log['outlier_rows']}")
print("\nCleaned county row counts:")
print(clean_txns["county"].value_counts(dropna=False))

=== Data-Quality Audit Table ===


,issue,dimension,column,rows_affected
0,Duplicate txn_id rows,Uniqueness,txn_id,12
1,Non-standard casing or trailing whitespace in ...,Consistency,txn_type,384
2,Slash date delimiter (YYYY/MM/DD instead of YY...,Validity,txn_date,5
3,Missing (NaN) transaction amounts,Completeness,amount,22
4,Non-numeric text ('N/A') in transaction amounts,Validity,amount,0
5,"Extreme KES 250,000 transaction amounts",Accuracy,amount,5
6,Unregistered agent_id ('AG099') not in agents ...,Consistency,agent_id,8



=== Post-Merge Validation ===
Raw transactions rows:     612
Cleaned transactions rows: 600 (duplicates removed: 12)
Unknown agent txns (raw):  8
Unknown agent txns (clean):7 (assigned county='Unknown')
Imputed amount rows:       20 | Medians by type: {'Deposit': 1803.5, 'Withdrawal': 1750.5}
IQR upper fence:           5513.75 | Flagged outliers: 53

Cleaned county row counts:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


### Cleaning Decisions, Row Counts & Justifications
* **Uniqueness & Row Counts:** Raw `agent_transactions.csv` has **612** rows; removing **12** duplicate `txn_id` rows (`keep="first"`) leaves **600** unique transactions.
* **Consistency (`txn_type`):** Trimming whitespace and applying title casing standardises all rows to **Deposit** (**373**) and **Withdrawal** (**227**).
* **Validity & Types:** `amount` is converted to numeric (`float64`), and slash-delimited `txn_date` strings (`YYYY/MM/DD`) are parsed into ISO `YYYY-MM-DD` with **0** unparseable dates.
* **Completeness (`amount_imputed`):** Rows with missing (`NaN`) or non-numeric (`"N/A"`) amounts are flagged with `amount_imputed = True` and imputed using each `txn_type`'s median (**KES 1,803.50** for `Deposit`, **KES 1,750.50** for `Withdrawal`) so valid transaction rows are preserved without skewing averages.
* **Outliers (`is_outlier`) & `KES 250,000` Transactions:** Using `1.5 * IQR` (`Q1 = 1,105.0`, `Q3 = 2,868.5`, `IQR = 1,763.5`, upper fence `5,513.75`), **53** transactions are flagged in `is_outlier = True` (including **5** transactions of **KES 250,000**). Rather than deleting them automatically, we flag them for finance, float, and AML/compliance review to distinguish legitimate bulk merchant/liquidity movements from data-entry errors (such as an extra zero).
* **Merge & Unknown Agents:** A left join on `agent_id` keeps all **600** deduplicated transactions; **7** deduplicated rows (**8** in raw data) reference unregistered `AG099` and receive `county = "Unknown"`.

In [3]:
# (a) County-level summary with total value, number of transactions, and average value by txn_type
county_summary = (
    clean_txns.groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        txn_count=("txn_id", "count"),
        avg_value=("amount", "mean"),
    )
    .reset_index()
)
county_summary["avg_value"] = county_summary["avg_value"].round(2)

print("=== County Summary Split by Transaction Type ===")
display(county_summary)

# (b) Weekly pivot table of transaction value by county
clean_txns_wk = clean_txns.copy()
clean_txns_wk["week_start"] = (
    pd.to_datetime(clean_txns_wk["txn_date"])
    .dt.to_period("W")
    .apply(lambda p: p.start_time.strftime("%Y-%m-%d"))
)
weekly_pivot = clean_txns_wk.pivot_table(
    index="county",
    columns="week_start",
    values="amount",
    aggfunc="sum",
    fill_value=0.0,
    margins=True,
    margins_name="Total",
)

print("\n=== Weekly Transaction Value Pivot Table by County (KES) ===")
display(weekly_pivot)

=== County Summary Split by Transaction Type ===


,county,txn_type,total_value,txn_count,avg_value
0,Kiambu,Deposit,334737.0,43,7784.58
1,Kiambu,Withdrawal,77024.5,27,2852.76
2,Kisumu,Deposit,436501.5,74,5898.67
3,Kisumu,Withdrawal,76348.5,37,2063.47
4,Mombasa,Deposit,149094.0,72,2070.75
5,Mombasa,Withdrawal,368592.0,61,6042.49
6,Nairobi,Deposit,543698.0,118,4607.61
7,Nairobi,Withdrawal,161932.0,62,2611.81
8,Nakuru,Deposit,184944.0,60,3082.40
9,Nakuru,Withdrawal,328568.5,39,8424.83



=== Weekly Transaction Value Pivot Table by County (KES) ===


week_start,2026-02-23,2026-03-02,2026-03-09,2026-03-16,2026-03-23,2026-03-30,2026-04-06,2026-04-13,2026-04-20,2026-04-27,Total
county,,,,,,,,,,,
Kiambu,0.0,18317.0,267745.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0,411761.5
Kisumu,1893.0,287933.0,23186.0,28126.0,27077.0,41077.0,15341.0,29470.0,37439.0,21308.0,512850.0
Mombasa,6759.0,285696.5,30504.5,25171.5,17050.0,38344.0,47119.5,26730.0,16687.0,23624.0,517686.0
Nairobi,7046.0,68967.0,52887.5,43134.5,41400.5,282931.0,56813.5,56406.5,65135.0,30908.5,705630.0
Nakuru,2770.5,56574.0,28730.0,39795.0,18972.5,18145.0,276609.0,26179.5,30862.0,14875.0,513512.5
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0,27156.0
Total,18468.5,717487.5,403053.5,159920.0,116004.0,411613.0,420737.0,151051.0,176507.0,113754.5,2688596.0


## 2. County Summary (by Transaction Type) & Weekly Pivot Table

## 3. Data Quality Report

### Six-Dimension Data Quality Assessment
* **Completeness:** The raw extract (`612` rows) had missing values in `amount`, and `20` unique transactions (`NaN` or `"N/A"`) lacked numeric amounts prior to cleaning. These are flagged via `amount_imputed = True` and median-imputed by `txn_type` so no transaction records are lost.
* **Uniqueness:** Identified **12** duplicate `txn_id` rows (`24` total rows involved in duplicate IDs) and deduplicated (`keep="first"`) to **600** unique transactions.
* **Consistency:** `txn_type` contained mixed casing and trailing whitespace (`"DEPOSIT "`, `"deposit"`, `"withdrawal"`), standardised to `"Deposit"` and `"Withdrawal"`. In addition, **7** deduplicated rows (**8** raw rows) referenced `"AG099"`, which was missing from `agents.csv`, and were assigned `county = "Unknown"`.
* **Validity:** `amount` included non-numeric `"N/A"` strings and `txn_date` contained **5** slash-formatted dates (`YYYY/MM/DD`). Numeric parsing and ISO `YYYY-MM-DD` date standardisation resolved all format inconsistencies with **0** unparseable dates.
* **Accuracy:** Lognormal transaction amounts plus **5** injected **KES 250,000** entries resulted in **53** IQR outliers (`is_outlier = True`, above `KES 5,513.75`). Extreme amounts are retained with a flag for business/compliance verification rather than silently dropped.
* **Timeliness:** All parsed transactions fall within the expected 60-day window from **2026-03-01** to **2026-04-29**, with no future or out-of-range dates.

### Privacy & Kenya Data Protection Act (2019)
If the transaction extract also included **customer phone numbers** (especially when paired with transaction timestamps, amounts, or agent IDs), those phone numbers would constitute **personal data** under the **Kenya Data Protection Act (2019)** because they can directly or indirectly identify an individual mobile-money subscriber. Under the Act, we would protect them by applying **data minimisation and pseudonymisation/hashing** (replacing raw phone numbers with keyed cryptographic hashes or omitting them from analytical tables where only county/agent aggregates are needed), restricting access via **role-based access controls (RBAC)** and encryption in transit and at rest, enforcing **retention limits** so raw identifiers are not kept longer than necessary, and ensuring processing aligns with a **lawful, specified purpose** supported by appropriate governance and Data Protection Impact Assessment (DPIA) safeguards.